# A3-M2-2 · 실습 과제
02 노트북을 먼저 실행해 faq_store/ 폴더가 있어야 합니다. 준비 셀부터 순서대로 실행합니다.

## 준비
모든 과제가 함께 쓰는 준비 셀입니다. 02 노트북을 먼저 실행해 faq_store/가 있어야 합니다.

In [1]:
import json, time
import numpy as np
import pandas as pd
import faiss
from emb import embed, EMB_MODEL, RERANK_MODEL

index = faiss.read_index("faq_store/index.faiss")
meta = pd.read_csv("faq_store/meta.csv").set_index("faq_id")
man = json.load(open("faq_store/manifest.json", encoding="utf-8"))
assert man["model"] == EMB_MODEL, "다른 모델로 만든 인덱스입니다"
tests = pd.read_csv("shop_faq_tests.csv")

def ids_of(categories):
    return meta.index[meta["category"].isin(categories)].to_numpy(dtype="int64")

def search(question, k=3, categories=None):
    params = None
    if categories:
        params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_of(categories)))
    D, I = index.search(embed([question], kind="query"), k, params=params)
    hits = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i != -1]
    res = meta.loc[[i for i, _ in hits], ["category", "question", "answer"]].copy()
    res.insert(0, "score", [round(s, 3) for _, s in hits])
    return res

print("FAQ", index.ntotal, "건 | 평가 질문", len(tests), "개 | 모델", man["model"])
from transformers import pipeline
rr = pipeline("text-classification", model=RERANK_MODEL, device=-1)

def search_rerank(question, k=3, pool=8, categories=None):
    cand = search(question, k=pool, categories=categories)              # 1단계: FAISS로 후보
    pairs = [{"text": question, "text_pair": f"{r.question} {r.answer}"} for r in cand.itertuples()]
    cand["rerank"] = [round(s["score"], 3) for s in rr(pairs, function_to_apply="none")]
    return cand.sort_values("rerank", ascending=False).head(k)          # 2단계: 다시 줄 세우기

def evaluate(fn, **kw):
    rows = []
    for q, gold in tests.itertuples(index=False):
        ranked = fn(q, **kw).index.tolist()
        rows.append({"question": q, "gold": gold, "top1": ranked[0],
                     "rank": ranked.index(gold) + 1 if gold in ranked else np.nan})
    return pd.DataFrame(rows)

def summary(ev):
    return {"Hit@1": round((ev["rank"] == 1).mean(), 3), "Hit@3": round((ev["rank"] <= 3).mean(), 3),
            "MRR": round((1 / ev["rank"]).fillna(0).mean(), 3)}

FAQ 20 건 | 평가 질문 12 개 | 모델 intfloat/multilingual-e5-small


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1531.34it/s]


## 과제 1 · 카테고리를 알면 얼마나 좋아지나
평가 질문마다 정답 FAQ의 카테고리로 필터를 걸고(`categories=[...]`) 검색해서, 필터 없을 때와 Hit@1·MRR을 비교하세요. 화면에서 고객이 카테고리를 먼저 고르는 상황입니다.

In [2]:
rows = []
for q, gold in tests.itertuples(index=False):
    cat = meta.loc[gold, "category"]
    ranked = search(q, k=5, categories=[cat]).index.tolist()
    rows.append({"question": q, "gold": gold, "top1": ranked[0], "rank": ranked.index(gold) + 1 if gold in ranked else np.nan})
ev_cat = pd.DataFrame(rows)
print(pd.DataFrame({"필터 없음": summary(evaluate(search, k=5)), "정답 카테고리 필터": summary(ev_cat)}).to_string())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1328.59it/s]


       필터 없음  정답 카테고리 필터
Hit@1  0.917         1.0
Hit@3  1.000         1.0
MRR    0.944         1.0


## 과제 2 · 리랭킹 후보 수(pool) 고르기
`search_rerank`의 `pool`을 3, 5, 8, 20으로 바꿔 Hit@1·MRR과 질문당 시간을 표로 만드세요. 후보가 많을수록 항상 좋은지 확인합니다.

In [3]:
rows = []
for pool in [3, 5, 8, 20]:
    t = time.perf_counter()
    ev = evaluate(search_rerank, k=min(5, pool), pool=pool)
    ms = (time.perf_counter() - t) / len(tests) * 1000
    rows.append({"pool": pool, **summary(ev), "ms/질문": round(ms)})
print(pd.DataFrame(rows).to_string(index=False))

 pool  Hit@1  Hit@3   MRR  ms/질문
    3  0.917    1.0 0.958    251
    5  0.833    1.0 0.917    371
    8  0.833    1.0 0.917    608
   20  0.833    1.0 0.917   1535


## 과제 3 · 평가 질문 늘리기
고객이 실제로 쓸 법한 질문 5개를 직접 만들어 정답 FAQ 번호와 함께 `tests`에 추가하고, 검색만/검색+리랭킹 결과를 다시 비교하세요. 평가 질문이 늘면 숫자가 어떻게 바뀌나요?

In [4]:
new = pd.DataFrame([
    ("주문한 거 없던 걸로 해 주세요", 501), ("로그인 암호가 생각이 안 나요", 601), ("쿠폰이랑 포인트 같이 써도 돼요?", 702),
    ("마음이 바뀌어서 돌려보내면 택배비 내야 해요?", 302), ("현금영수증 받고 싶어요", 203),
], columns=tests.columns)
print(new.assign(faq_question=meta.loc[new["gold_faq_id"], "question"].values).to_string(index=False))
tests = pd.concat([tests, new], ignore_index=True)
print("평가 질문", len(tests), "개")
print(pd.DataFrame({"검색만": summary(evaluate(search, k=5)), "검색+리랭킹": summary(evaluate(search_rerank, k=5, pool=8))}).to_string())

                 question  gold_faq_id          faq_question
        주문한 거 없던 걸로 해 주세요          501         주문을 취소하고 싶어요.
         로그인 암호가 생각이 안 나요          601         비밀번호를 잊어버렸어요.
       쿠폰이랑 포인트 같이 써도 돼요?          702  쿠폰과 적립금을 같이 쓸 수 있나요?
마음이 바뀌어서 돌려보내면 택배비 내야 해요?          302 단순 변심 반품 배송비는 누가 내나요?
             현금영수증 받고 싶어요          203     현금영수증은 어떻게 발급하나요?
평가 질문 17 개
         검색만  검색+리랭킹
Hit@1  0.824   0.824
Hit@3  0.941   1.000
MRR    0.873   0.912
